In [1]:
import kagglehub
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
from collections import Counter

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/home/onnikiv/miniconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:
path = kagglehub.dataset_download("rohitgr/wikitext")
print(path)

DATA_PATH = None
for root, _, files in os.walk(path):
    for f in files:
        if "train" in f and f.endswith(".tokens"):
            DATA_PATH = os.path.join(root, f)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read().lower().replace("\n", " ")

words = text.split()

words = words[:200_000]

/home/onnikiv/.cache/kagglehub/datasets/rohitgr/wikitext/versions/1


In [3]:
MAX_VOCAB = 8000

vocab_counts = Counter(words).most_common(MAX_VOCAB)

vocab = {w:i for i,(w,_) in enumerate(vocab_counts)}
inv_vocab = {i:w for w,i in vocab.items()}

tokens = [vocab[w] for w in words if w in vocab]

vocab_size = len(vocab)
print("vocab:", vocab_size)

vocab: 8000


In [4]:
class WikiDataset(Dataset):
    def __init__(self, tokens, seq_len):
        self.tokens = torch.tensor(tokens, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, i):
        x = self.tokens[i:i+self.seq_len]
        y = self.tokens[i+1:i+self.seq_len+1]
        return x, y

In [5]:
SEQ_LEN = 24
BATCH_SIZE = 32

dataset = WikiDataset(tokens, SEQ_LEN)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

In [6]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, d_model)

        self.attn = nn.MultiheadAttention(d_model, 2, batch_first=True)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)

        seq = x.size(1)
        mask = torch.triu(torch.ones(seq, seq, device=x.device), diagonal=1).bool()

        attn, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + attn)

        x = self.norm2(x + self.ff(x))

        return self.fc(x)

In [7]:
model = MiniGPT(vocab_size, d_model=64).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

print("ready")

ready


In [8]:
EPOCHS = 1 

for epoch in range(EPOCHS):
    model.train()
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        out = model(x)

        loss = criterion(out.view(-1, vocab_size), y.view(-1))

        loss.backward()
        optimizer.step()

        total += loss.item()

    print("loss:", total / len(loader))

loss: 5.59180079735979


In [9]:
def perplexity(loss):
    return torch.exp(torch.tensor(loss))

print("perplexity:", perplexity(total / len(loader)).item())

perplexity: 268.2181701660156


In [10]:
def generate(model, start_text, max_len=20):
    model.eval()

    tokens = [vocab.get(w, 0) for w in start_text.lower().split()]
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_len):
        with torch.no_grad():
            out = model(x)

            probs = torch.softmax(out[:, -1, :], dim=-1)

            # top-k sampling (kevyt versio)
            topk = torch.topk(probs, k=5, dim=-1)
            idx = torch.randint(0, 5, (1,))
            next_token = topk.indices[0, idx].item()

        x = torch.cat([x, torch.tensor([[next_token]], device=device)], dim=1)

    return x.squeeze().tolist()

In [11]:
def decode(tokens):
    return " ".join([inv_vocab.get(t, "<UNK>") for t in tokens])

In [12]:
seed = "machine learning is"
output = generate(model, seed, max_len=15)

print(decode(output))

machine learning is not . = early 20th century ad . in a result . in a result
